In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score
import numpy as np

# Fill missing text with empty string
train_text = train.copy()
val_text = val.copy()
for c in ["clinical_notes", "call_center_notes"]:
    train_text[c] = train_text[c].fillna("")
    val_text[c] = val_text[c].fillna("")

X_train = train_text[num_cols + cat_cols + ["clinical_notes","call_center_notes"]]
y_train = train_text["fraud_label"]
X_val   = val_text[num_cols + cat_cols + ["clinical_notes","call_center_notes"]]
y_val   = val_text["fraud_label"]

text_features = ["clinical_notes","call_center_notes"]

preprocess_with_text = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ("txt", TfidfVectorizer(
            max_features=5000,
            ngram_range=(1,2),
            min_df=2
        ), "clinical_notes")
    ],
    remainder="drop"
)

# NOTE: ColumnTransformer can't apply one TFIDF to 2 text cols directly in this layout.
# We'll combine the text cols into one.
train_text["combined_notes"] = train_text["clinical_notes"] + " " + train_text["call_center_notes"]
val_text["combined_notes"] = val_text["clinical_notes"] + " " + val_text["call_center_notes"]

X_train2 = train_text[num_cols + cat_cols + ["combined_notes"]]
X_val2   = val_text[num_cols + cat_cols + ["combined_notes"]]

preprocess_with_text = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat_cols),
        ("txt", TfidfVectorizer(
            max_features=5000,
            ngram_range=(1,2),
            min_df=2
        ), "combined_notes"),
    ]
)

model_text = Pipeline([
    ("prep", preprocess_with_text),
    ("model", HistGradientBoostingClassifier(max_depth=6, learning_rate=0.05, max_iter=300))
])

model_text.fit(X_train2, y_train)
proba_text = model_text.predict_proba(X_val2)[:,1]

print("TFIDF+HGB PR-AUC:", average_precision_score(y_val, proba_text))

for pct in [0.05, 0.10, 0.20]:
    k = max(1, int(pct * len(y_val)))
    top_idx = np.argsort(-proba_text)[:k]
    recall_k = y_val.iloc[top_idx].sum() / y_val.sum() if y_val.sum() > 0 else 0
    precision_k = y_val.iloc[top_idx].mean()
    print(f"Top {int(pct*100)}% k={k} | Precision={precision_k:.3f} | Recall={recall_k:.3f}")
Why you got this error (simple)

TfidfVectorizer creates a sparse matrix (to save memory).
But HistGradientBoostingClassifier in sklearn requires dense input, so it throws:

“Sparse data was passed… but dense data is required.”

✅ Fix: use a model that supports sparse (best choice for TF-IDF):
	•	LogisticRegression
	•	LinearSVC
	•	SGDClassifier
	•	(or convert to dense, but that’s not ideal)

We’ll do the professional fix: TF-IDF + Logistic Regression (sparse-friendly and interview-perfect).

⸻

Step 8 (Fixed): TF-IDF + Logistic Regression (works with sparse)
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

# Prepare text
train_text = train.copy()
val_text = val.copy()

for c in ["clinical_notes", "call_center_notes"]:
    train_text[c] = train_text[c].fillna("")
    val_text[c] = val_text[c].fillna("")

train_text["combined_notes"] = train_text["clinical_notes"] + " " + train_text["call_center_notes"]
val_text["combined_notes"] = val_text["clinical_notes"] + " " + val_text["call_center_notes"]

X_train2 = train_text[num_cols + cat_cols + ["combined_notes"]]
y_train2 = train_text["fraud_label"]
X_val2   = val_text[num_cols + cat_cols + ["combined_notes"]]
y_val2   = val_text["fraud_label"]

preprocess_tfidf = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
        ("txt", TfidfVectorizer(
            max_features=8000,
            ngram_range=(1,2),
            min_df=2
        ), "combined_notes"),
    ]
)

tfidf_lr = Pipeline([
    ("prep", preprocess_tfidf),
    ("model", LogisticRegression(
        solver="saga",
        penalty="l2",
        C=1.0,
        max_iter=8000,
        class_weight="balanced",
        n_jobs=-1
    ))
])
tfidf_lr.fit(X_train2, y_train2)
proba_tfidf = tfidf_lr.predict_proba(X_val2)[:, 1]

print("TFIDF+LR PR-AUC:", average_precision_score(y_val2, proba_tfidf))

for pct in [0.05, 0.10, 0.20]:
    k = max(1, int(pct * len(y_val2)))
    top_idx = np.argsort(-proba_tfidf)[:k]
    recall_k = y_val2.iloc[top_idx].sum() / y_val2.sum() if y_val2.sum() > 0 else 0
    precision_k = y_val2.iloc[top_idx].mean()
    print(f"Top {int(pct*100)}% k={k} | Precision={precision_k:.3f} | Recall={recall_k:.3f}")
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import average_precision_score

# Ensure combined notes exist
train_text = train.copy()
val_text = val.copy()
for c in ["clinical_notes", "call_center_notes"]:
    train_text[c] = train_text[c].fillna("")
    val_text[c] = val_text[c].fillna("")
train_text["combined_notes"] = train_text["clinical_notes"] + " " + train_text["call_center_notes"]
val_text["combined_notes"] = val_text["clinical_notes"] + " " + val_text["call_center_notes"]

X_train2 = train_text[num_cols + cat_cols + ["combined_notes"]]
y_train2 = train_text["fraud_label"]
X_val2   = val_text[num_cols + cat_cols + ["combined_notes"]]
y_val2   = val_text["fraud_label"]

preprocess_tfidf = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
        ("txt", TfidfVectorizer(
            max_features=8000,
            ngram_range=(1,2),
            min_df=2
        ), "combined_notes"),
    ]
)

sgd = Pipeline([
    ("prep", preprocess_tfidf),
    ("model", SGDClassifier(
        loss="log_loss",          # gives predict_proba
        alpha=1e-4,
        class_weight="balanced",
        max_iter=5000,
        tol=1e-3,
        random_state=42
    ))
])

sgd.fit(X_train2, y_train2)
proba_sgd = sgd.predict_proba(X_val2)[:, 1]

print("TFIDF+SGD PR-AUC:", average_precision_score(y_val2, proba_sgd))

for pct in [0.05, 0.10, 0.20]:
    k = max(1, int(pct * len(y_val2)))
    top_idx = np.argsort(-proba_sgd)[:k]
    recall_k = y_val2.iloc[top_idx].sum() / y_val2.sum()
    precision_k = y_val2.iloc[top_idx].mean()
    print(f"Top {int(pct*100)}% k={k} | Precision={precision_k:.3f} | Recall={recall_k:.3f}")

print("proba min/max:", float(np.min(proba_sgd)), float(np.max(proba_sgd)))